# Lesson 6: Mapping Emotions

## Overview

This is the culminating lesson of the unit. Over the last several lessons you have:

- **Lesson 3** — Loaded and cleaned raw Reddit data
- **Lesson 4** — Extracted place names using NER models and resolved them to coordinates with a geoparser
- **Lesson 5** — Scored each sentence for sentiment using VADER and RoBERTa

Now you have two things attached to each sentence: a **location** and an **emotion**. In this lesson you will aggregate those scores by place and put them on a map — and then critically evaluate what that map can and cannot tell you.

---


## 1. Load the Data

Load the sentiment dataset your team produced in Lesson 5.2. If the file is not in memory, load it from the pickle file your teammate committed to the repository.


In [5]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import pearsonr
import numpy as np

# ── 1. Load sentence-level sentiment data ───────────────────────────────────
try:
    df_reddit_sentiment_full
    print("✅ df_reddit_sentiment_full is already loaded in memory")
except NameError:
    print("📥 Loading from shared data folder...")
    try:
        df_reddit_sentiment_full = pd.read_pickle('data/jmu_reddit_sentiment_full.pickle')
        print(f"✅ Loaded {len(df_reddit_sentiment_full):,} rows")
    except FileNotFoundError:
        print("❌ File not found — check that a teammate completed Section 7 of Lesson 5.2")

# ── 2. Aggregate to place-level summary ─────────────────────────────────────
df_reddit_place_sentiments = (
    df_reddit_sentiment_full
    .dropna(subset=['place', 'latitude', 'longitude'])
    .astype({'latitude': float, 'longitude': float})
    .groupby('place', sort=False)
    .agg(
        location_count=('place', 'size'),
        latitude=('latitude', 'first'),
        longitude=('longitude', 'first'),
        sentences=('sentences', lambda x: ' | '.join(str(s) for s in list(x)[:5])),
        avg_roberta_compound=('roberta_compound', 'mean'),
    )
    .reset_index()
)

# ── 3. Merge place_type from the reviewed geoparsed data ────────────────────
_ref = (pd.read_csv('data/jmu_reddit_geoparsed_long.csv', usecols=['place', 'place_type'])
          .drop_duplicates('place'))
df_reddit_place_sentiments = df_reddit_place_sentiments.merge(_ref, on='place', how='left')

print(f"\n📍 {len(df_reddit_place_sentiments):,} unique places")
print(f"   place_type coverage: "
      f"{df_reddit_place_sentiments['place_type'].notna().sum()} / "
      f"{len(df_reddit_place_sentiments)} labelled")
print(f"\nTop types:\n{df_reddit_place_sentiments['place_type'].value_counts(dropna=False).head(8).to_string()}")

df_reddit_sentiment_full.sample(5, random_state=42)


📥 Loading from shared data folder...
✅ Loaded 1,785 rows


FileNotFoundError: [Errno 2] No such file or directory: 'data/jmu_reddit_geoparsed_long.csv'

## 2. The Pipeline — A Critical Review

Before we visualize anything, it is worth stepping back to review what each stage of the pipeline actually did — and where it could have gone wrong.

### Lesson 3: Data Cleaning
You loaded raw Reddit posts and split them into individual sentences. You cleaned up encoding issues, removed very short strings, and filtered out noise. This early stage determines what the data will look like later.

> **Limitation:** Sentence splitting is imperfect. A sentence that spans a line break or uses non-standard punctuation may have been cut in the wrong place, which would mislead the sentiment model.

### Lesson 4: Location Extraction
You ran two NER (Named Entity Recognition) models — spaCy and a transformer-based tagger — to identify place names in each sentence, then used a geoparser to convert those names to coordinates.

> **Limitation:** NER models confuse place names with other entities (people, organizations, common words). The geoparser resolves ambiguous names by population rank and other weights, which means "London" will almost always map to the UK even if another location is closer and the author the author meant something else. You corrected some of these manually — but not all of them.

### Lesson 5: Sentiment Analysis
You ran VADER (rule-based, fast) and RoBERTa (transformer, context-aware) on each sentence and compared their outputs. You found contradictions — sentences where the two models disagreed — and used them to understand each model's blind spots.

> **Limitation:** Both models were trained on general social media text. Reddit language, irony, sarcasm, and in-group references may confuse either model. Sentiment scores are averages across sentences — a place mentioned once in a very negative post will look "negative" even if 99% of posts about it are neutral.




## 3. Fuzzy Data and Critical Design Descisions

There is a law of diminishing returns when cleaning up the geoparsed data. At some point, it is no longer worth parsing each individual location and you have to accept that some of the data will be innacurate. Likewise, the sentiment analyzer is going to give a rough estimate, but will not give a 100% accurate picture of all the emotions around each location. There is an inherent fuzziness about the way humans talk about locations and the emotions they attach to them. Indeed, the phrase "I love JMU!" is an emotion about a location that seems very definite, but even here it may only be one particular space in JMU you love and at one particular time. Since the data is so careful we have to be very critical in how we visualize it.

Modern mapping technologies like plotly and mapbox allow us to create maps very quickly. If we do not consider the underlying data, we can come up with two radically different representations. Consider the two maps below. Each takes data we created about JMU and puts it on a map. Consider what conclusion you would draw from each map.

In [ ]:
import numpy as np


# ── Data prep ────────────────────────────────────────────────────────────
df_A = df_reddit_place_sentiments.copy()                          # Map A: everything
df_B = df_reddit_place_sentiments[
    df_reddit_place_sentiments['location_count'] >= 5             # Map B: high-confidence only
].copy()

# ── Map A: design choices that read as "students are negative" ───────────
#
# The key manipulation is the MISSING color_continuous_midpoint.
# Plotly anchors the neutral (yellow) colour at the midpoint of the
# data RANGE, not at zero.  If sentiment spans [-0.3, +0.6], the midpoint
# is +0.15 — so ANY place with sentiment below +0.15 (including mildly
# positive ones) renders red-to-yellow on the scale.
#
data_mid = (df_A['avg_roberta_compound'].min() + df_A['avg_roberta_compound'].max()) / 2
looks_negative = (df_A['avg_roberta_compound'] < data_mid).sum()


fig_A = px.scatter_map(
    df_A,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn",
    # ← NO color_continuous_midpoint: neutral colour drifts to the data midpoint
    size_max=100,                               # ← one place will dominate visually
    map_style="carto-darkmatter",               # ← dark basemap, ominous tone
    center={"lat": 39.0, "lon": -89.0},         # ← national framing
    zoom=3,                                     # ← geoparser errors fully visible
    height=620,
    title="<b>Map A</b>"
)
fig_A.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_A.show()

# ── Map B: design choices that read as "sentiment is mostly positive" ────

NEUTRAL_THRESHOLD = 0.10   # generous neutral band — ±0.10 around zero

bins       = [-float('inf'), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float('inf')]
cat_labels = ['Negative', 'Neutral', 'Positive']
df_B['sentiment_category'] = pd.cut(
    df_B['avg_roberta_compound'], bins=bins, labels=cat_labels
)

# Quantile size classification: equal number of places per class
df_B['size_class'] = pd.qcut(
    df_B['location_count'], q=4, labels=[1, 2, 3, 4], duplicates='drop'
).astype(float)

neg = (df_B['sentiment_category'] == 'Negative').sum()
neu = (df_B['sentiment_category'] == 'Neutral').sum()
pos = (df_B['sentiment_category'] == 'Positive').sum()


fig_B = px.scatter_map(
    df_B,
    lat="latitude", lon="longitude",
    size="size_class",
    color="sentiment_category",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "size_class": False, "latitude": False, "longitude": False},
    color_discrete_map={'Negative': '#d62728', 'Neutral': '#aec7e8', 'Positive': '#2ca02c'},
    category_orders={"sentiment_category": cat_labels},
    size_max=40,
    map_style="carto-positron",                 # ← clean, neutral basemap
    center={"lat": 38.4, "lon": -79.0},         # ← Shenandoah Valley / campus context
    zoom=7,                                     # ← regional framing
    height=620,
    title="<b>Map B</b>"
)
fig_B.update_layout(margin={"r": 0, "t": 90, "l": 0, "b": 0})
fig_B.show()



NameError: name 'df_reddit_place_sentiments' is not defined

> 💡 **Reflection:** 
> - What does each map tell you? 
> - Why do you reach that conclusion? 
> - What are some visual distortions that each introduces?
> &nbsp;

## 4 Map Design

Map design is a highly specialized field in cartography with its own long history. For this lesson, we will cover three major design principles: symbolization, color theory, and scale. Each of these principles helps tell a particular story. 

- **Symbolization** - determines what marker or symbol you use to represent your data. This covers the number of symbols, their relative size, shape, and which data points are worth showing at all.

- **Color theory** - determines how you are going to differentiate different elements on the map through your color choices. This includes how you distinguish between symbols, but also how you contrast the base map style with the data projected onto it. One major consideration for color theory is always accessibility. How do you use colors effectively so that a maximum number of users can understand your story?

- **Scale** - determines the zoom and viewport of your map. This may seem simple, but you have to consider the conceptual scale of your story. Are you telling a story about Harrisonburg, Virginia, the US, or even the entire world? Determining the scale of the story makes the zoom far more consequential.

Each of these principles has been broken down into mini-units on each element below. **Run each code cell, compare the versions, and answer the reflection questions before moving on.** Do not proceed to the design brief until you have worked through all eight.

| Principle | # | Design Decision | The core question |
|---|---|---|---|
| **Symbolization** | 1 | **Filtering threshold** | Which places are worth showing? |
| &nbsp; | 2 | **Place type filter** | Should Country and City sentiment appear on the same map? |
| &nbsp; | 3 | **Bubble size** | What does size mean, and does it serve your argument? |
| &nbsp; | 4 | **Size classification** | Equal interval, quantile, or Jenks natural breaks? |
| &nbsp; | 5 | **Sentiment bucketing** | Continuous gradient or discrete categories? |
| **Color theory** | 6 | **Color scale** | Which scale is honest *and* accessible? |
| &nbsp; | 7 | **Base map style** | What tone and context does the basemap set? |
| **Scale** | 8 | **Zoom & viewport** | What context do you show — and what do you hide? |

### Decision 1: Filtering Threshold

The data is very wide ranging. Some locations appear only once while others appear many times (i.e. Harrisonburg). While a "true" map would show all the data, displaying each individual location  can be visually distracting and make it much harder to see patterns. You can filter out this noise by adjusting the minimum count for each location. Thus, if a location only appears 2 times it might not be all that relevant. 

The tricky thing is that there is no concrete rule for picking a threshold. Instead it depends on a host of factors. If you are doing an analysis of the buildings mentioned in each data set they may not be mentioned all that often so a lower threshold is necessary. Alternatively, if you are doing an analysis of cities, mentioned the threshold may be higher because they tend to be named more often. The sweet spot is one that shows enough data to tell the story without crowding the map, while also not erasing data that could contradict your story. 

You will have to justify your threshold in your project, so consider this carefully.



In [12]:
# Decision 1: Filtering threshold — how much data is enough?
# ← Change this value and re-run the cell to see the effect
min_count = 1

subset = df_reddit_place_sentiments[
    df_reddit_place_sentiments['location_count'] >= min_count
].copy()

print(f"── min_count ≥ {min_count}: {len(subset)} locations, "
      f"{subset['location_count'].sum()} total posts ──")

fig = px.scatter_map(
    subset,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    size_max=40,
    map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0},
    zoom=6, height=430,
    title=f"Filter: min post count ≥ {min_count}  →  {len(subset)} locations visible"
)
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
fig.show()




── min_count ≥ 1: 522 locations, 1785 total posts ──


Critical question: What do you think is the "sweet spot" for eliminating the minimum count? Why do you think that is?

### Decision 2: Place Type Filter

Locations exist at different scales and are therefore different in kind. The location "United States" is a country, while "Harrisonburg" is a city. Technically, they both exist on the same map, but if you are trying to tell a story about how JMU students feel about their community country-level information might not be all that relevant. Conversely, if you are considering how JMU feels about other places in Virginia then knowing how students feel about D-Hall is not all that relevant.

In lesson 4 we cleaned up the geoparsed `place_types` that made these distinctions. One way we can reduce map distortion is making sure that all the symbols were are looking at are the ones we need for our particular story.

We can filter for specific types by adding those types to the `place_types` list.

```
place_types = ['City', 'Building']
```

This filter will only show Cities and Buildings. The options include:

`'Country', 'State', 'City', 'Neighborhood','Building', , 'University', 'Road', 'Region'` 

In [14]:
# Decision 2: Place type filter — which geographic scales belong in this story?
# ← Change these values and re-run the cell to see the effect
min_count   = 3
place_types = ['City', 'Building']   # set to None to show all types

# Named place_type options:
#   'Country', 'State', 'Region', 'City', 'Neighborhood',
#   'University', 'Road', 'Building', 'Natural Feature'

if place_types is None:
    subset = df_reddit_place_sentiments[
        df_reddit_place_sentiments['location_count'] >= min_count
    ].copy()
else:
    subset = df_reddit_place_sentiments[
        (df_reddit_place_sentiments['location_count'] >= min_count) &
        df_reddit_place_sentiments['place_type'].isin(place_types)
    ].copy()

type_counts = subset['place_type'].value_counts(dropna=False).to_dict()
print(f"── min_count ≥ {min_count}, place_types = {place_types}: {len(subset)} places  {type_counts} ──")

fig = px.scatter_map(
    subset,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"place_type": True, "location_count": True,
                "avg_roberta_compound": ":.3f",
                "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    size_max=40,
    map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0},
    zoom=5, height=450,
    title=f"Place type filter: {place_types}  →  {len(subset)} locations"
)
fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
fig.show()

print("\n💡 Try place_types = None (all), ['City'], ['City', 'State'], ['City', 'Building'], ['Natural Feature'].")
print("   Which countries appear when you use None? Do they reflect real travel or geoparser errors?")
print("   Which filter tells the most honest story for your research question?")


KeyError: 'place_type'

### Decision 3: Bubble Size

On a bubble map, the reader's eye is drawn to the largest circles first. **Size is an argument** — making something bigger implies it matters more. That means the moment you decide what drives bubble size, you have made a claim about what is important.

The default choice — `size = post count` — argues that places discussed more are more significant. That is often reasonable, but it creates a problem: a single heavily-discussed location can visually dominate the entire map, even if its sentiment is unremarkable.

| Encoding | What it emphasises | Risk |
|---|---|---|
| `size = raw count` (linear) | Frequency of discussion | One outlier can overwhelm everything else |
| `size = √count` (square root) | Frequency, compressed | Scale is less intuitive; readers may not know how to decode it |
| Uniform size | Nothing — only colour speaks | Makes it harder to see where the data is sparse vs. rich |

> 💡 **Questions to consider:**
> - With raw count, which location dominates? Is that the most important story in the data?
> - With uniform size, does the map become clearer or more confusing?
> - Is it ever *misleading* to use size as a redundant encoding (showing count *and* colour shows sentiment, but a large neutral bubble draws more attention than a small but strongly negative one)?

In [ ]:
# Decision 3: Bubble size — what does SIZE encode?

import numpy as np

df_size = df_reddit_place_sentiments.copy()
df_size['sqrt_count'] = np.sqrt(df_size['location_count'])
df_size['uniform']    = 1

base = dict(
    lat="latitude", lon="longitude",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"location_count": True, "avg_roberta_compound": ":.3f",
                "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=460
)

for size_col, size_max, label in [
    ("location_count", 60, "A — size = raw count (linear): popular places visually dominate"),
    ("sqrt_count",     35, "B — size = √count: compressed scale, less outlier dominance"),
    ("uniform",        12, "C — uniform size: only colour carries information"),
]:
    fig = px.scatter_map(df_size, size=size_col, size_max=size_max, title=label, **base)
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("💡 In version A, which location dominates visually? "
      "Is that dominance the story you want to tell — or does it distract from a more interesting finding?")


NameError: name 'df_reddit_place_sentiments' is not defined

### Decision 4: Size Classification — Equal Interval, Quantile, or Jenks?

Once you've decided that bubble size should encode post count, a second question follows immediately: **how do you translate raw counts into visual sizes?**

Reddit post counts are almost always **right-skewed**: a handful of well-known places (the university town, the state capital) accumulate dozens or hundreds of mentions, while the majority of locations appear only once or twice. If you map the raw value directly, one giant bubble dominates and every other place collapses toward invisible.

One solution is to **classify** counts into a small number of discrete size classes. The three standard classification methods are:

| Method | How it works | Best when... |
|---|---|---|
| **Equal interval** | Divides the count *range* into bins of equal width (e.g., 0–25, 25–50, 50–75, 75–100) | Data is roughly uniformly distributed — rare for geographic counts |
| **Quantile** | Divides the *sorted data* so each bin holds the same number of places (e.g., 25th / 50th / 75th percentile breaks) | You want every size class to appear equally often on the map |
| **Jenks natural breaks** | Finds the threshold values that **minimise within-class variance** — breaks fall at genuine gaps in the data | Data has natural clusters (almost always true for post-count data) |

**Why equal interval usually fails here:** If most places have 1–5 mentions and one has 200, all but one place fall into the lowest bin. The largest bin contains a single point — which defeats the purpose of classification entirely.

**Jenks is usually the right choice** for this data. It finds where the *real* gaps are — for example, it might identify {1–3, 4–10, 11–40, 41+} as the natural clusters rather than dividing the range arithmetically.

> 💡 **Questions to consider:**
> - After running the cell, how many places land in the largest equal-interval bin?
> - Where do the Jenks break points fall? Do those values correspond to any intuitive groupings?
> - Does the visual differentiation between small, medium, and large bubbles improve with Jenks?

In [ ]:
# Decision 4: Size classification — equal interval, quantile, Jenks natural breaks

import numpy as np

# First, examine the distribution of post counts
counts = df_reddit_place_sentiments['location_count']
print("── Post count distribution ──")
print(f"  min={counts.min()}, median={counts.median():.0f}, mean={counts.mean():.1f}, max={counts.max()}")
print(f"  Skewness: {counts.skew():.2f}  (>1 = strongly right-skewed)")
print(f"  Top 10 values: {sorted(counts.values, reverse=True)[:10]}")
print()

K = 4  # number of size classes — try changing to 3 or 5

# Method A: Equal interval — bins of equal WIDTH
df_ei = df_reddit_place_sentiments.copy()
print(f"── A: Equal Interval ({K} equal-width classes) ──")
print(pd.cut(counts, bins=K).value_counts().sort_index().to_string())
df_ei['size_class'] = pd.cut(counts, bins=K, labels=range(1, K + 1)).astype(float)

# Method B: Quantile — bins of equal COUNT (same number of places per class)
df_qt = df_reddit_place_sentiments.copy()
print(f"\n── B: Quantile ({K} classes, equal number of places per class) ──")
print(pd.qcut(counts, q=K, duplicates='drop').value_counts().sort_index().to_string())
df_qt['size_class'] = pd.qcut(counts, q=K, labels=range(1, K + 1), duplicates='drop').astype(float)

# Method C: Jenks natural breaks — bins at genuine gaps in the data
try:
    import mapclassify
    jnb = mapclassify.JenksNaturalBreaks(counts.values, k=K)
    df_jnb = df_reddit_place_sentiments.copy()
    df_jnb['size_class'] = (jnb.yb + 1).astype(float)
    print(f"\n── C: Jenks Natural Breaks ──")
    print(f"  Break points: {[round(b, 1) for b in jnb.bins]}")
    print(pd.Series(jnb.yb + 1).value_counts().sort_index()
            .rename(lambda i: f"Class {i}").to_string())
    c_label = "C — Jenks: breaks at genuine gaps in the data (minimises within-class variance)"
except ImportError:
    print("\n⚠️  mapclassify not installed. Run:  pip install mapclassify")
    print("   Showing Quantile as fallback for C.")
    df_jnb = df_qt.copy()
    c_label = "C — Jenks unavailable (mapclassify not installed); showing Quantile"

# Plot all three side by side
base = dict(
    lat="latitude", lon="longitude",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={"location_count": True, "size_class": False,
                "avg_roberta_compound": ":.3f", "latitude": False, "longitude": False},
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    size_max=50, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=460
)

for label, df_method in [
    ("A — Equal Interval: uniform bin width; dominated by skewed outliers", df_ei),
    ("B — Quantile: equal number of places per class; breaks may fall at arbitrary counts", df_qt),
    (c_label, df_jnb),
]:
    fig = px.scatter_map(df_method, size="size_class", title=label, **base)
    fig.update_layout(margin=dict(r=0, t=55, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("\n💡 In version A, how many places end up in the largest size class?")
print("   In Jenks, where do the break points fall — do those values make intuitive sense?")
print("   Which classification gives the most visual differentiation across the map?")


── Post count distribution ──
  min=1, median=1, mean=3.4, max=190
  Skewness: 11.70  (>1 = strongly right-skewed)
  Top 10 values: [np.int64(190), np.int64(153), np.int64(96), np.int64(73), np.int64(40), np.int64(27), np.int64(23), np.int64(21), np.int64(21), np.int64(19)]

── A: Equal Interval (4 equal-width classes) ──
location_count
(0.811, 48.25]     518
(48.25, 95.5]        1
(95.5, 142.75]       1
(142.75, 190.0]      2

── B: Quantile (4 classes, equal number of places per class) ──
location_count
(0.999, 2.0]    398
(2.0, 190.0]    124


ValueError: Bin labels must be one fewer than the number of bin edges

### Decision 5: Sentiment Bucketing — Continuous or Categorical?

RoBERTa produces a score from roughly −1 (strongly negative) to +1 (strongly positive). You have two main options:

**Continuous** — map the raw score directly to a colour gradient. Every decimal of precision is preserved and a reader can compare −0.12 to −0.34 by shade. The tradeoff is that subtle differences in colour are hard to read at a glance.

**Categorical (bucketed)** — assign each place to Negative / Neutral / Positive based on a threshold you choose. The map becomes immediately legible to any reader, but:

- **The threshold is itself a claim.** A threshold of ±0.05 will call many more places "Neutral" than a threshold of ±0.2. There is no objectively correct cutoff — it depends on what you want to argue.
- **A place scoring −0.051 looks identical to one scoring −0.9.** Both are "Negative." The distinction between mild and severe disappears.
- **Variation within a category is invisible.** Two cities can appear the same red even if one is barely negative and the other is strongly negative.

> 💡 **Questions to consider:**
> - Which version makes your main finding easier to communicate to a general audience?
> - Adjust `NEUTRAL_THRESHOLD` in the code below. At what value does a place you care about flip category?
> - Is continuous or categorical more *honest* given what you know about the underlying data quality?

In [ ]:
# Decision 5: Sentiment bucketing — continuous score vs. categorical labels

df_buck = df_reddit_place_sentiments.copy()

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    hover_name="place",
    hover_data={"avg_roberta_compound": ":.3f", "location_count": True,
                "latitude": False, "longitude": False},
    size_max=40, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=490
)

# Version A: continuous — every decimal of precision preserved
print("── Version A: Continuous ──")
fig_A = px.scatter_map(
    df_buck, color="avg_roberta_compound",
    color_continuous_scale="RdYlGn", color_continuous_midpoint=0,
    title="A — Continuous: precise, but requires careful reading",
    **base
)
fig_A.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig_A.show()

# Version B: categorical — adjust NEUTRAL_THRESHOLD and re-run
NEUTRAL_THRESHOLD = 0.05   # ← try 0.01, 0.1, 0.2 and observe what changes

bins       = [-float('inf'), -NEUTRAL_THRESHOLD, NEUTRAL_THRESHOLD, float('inf')]
cat_labels = ['Negative', 'Neutral', 'Positive']
df_buck['bucket'] = pd.cut(df_buck['avg_roberta_compound'], bins=bins, labels=cat_labels)

counts = df_buck['bucket'].value_counts().sort_index()
print(f"\n── Version B: Three buckets  (neutral = ±{NEUTRAL_THRESHOLD}) ──")
print(counts.to_string())

fig_B = px.scatter_map(
    df_buck, color="bucket",
    color_discrete_map={'Negative': '#d62728', 'Neutral': '#aec7e8', 'Positive': '#2ca02c'},
    category_orders={"bucket": cat_labels},
    title=f"B — Categorical (threshold ±{NEUTRAL_THRESHOLD}): readable, but variation within each bucket is hidden",
    **base
)
fig_B.update_layout(margin=dict(r=0, t=50, l=0, b=0))
fig_B.show()

print(f"\n💡 Change NEUTRAL_THRESHOLD to 0.2 and re-run. How many places flip to 'Neutral'? "
      "What story does that version of the map tell?")


── Version A: Continuous ──



── Version B: Three buckets  (neutral = ±0.05) ──
bucket
Negative    156
Neutral     214
Positive    152



💡 Change NEUTRAL_THRESHOLD to 0.2 and re-run. How many places flip to 'Neutral'? What story does that version of the map tell?


### Decision 6: Color Scale

Color is the most powerful — and most easily manipulated — variable on a sentiment map.

**Diverging vs. sequential:**
Diverging scales (red↔green, red↔blue) are appropriate when zero is a meaningful midpoint and values exist on both sides. Setting `color_continuous_midpoint=0` anchors the neutral colour at zero. Sequential scales (light→dark) work better when all values fall on one side — they are not appropriate here unless you are mapping only one sentiment category.

**Cultural associations and accessibility:**
- **Red/green (RdYlGn)** is immediately intuitive because red=danger and green=safe are deeply ingrained. But red-green color blindness affects roughly 8% of men — for those readers, the entire argument of your map is invisible.
- **Red/blue (RdBu)** is more accessible and carries less cultural baggage about "good" and "bad."
- **Viridis** is fully colorblind-safe and perceptually uniform, but loses the positive/negative framing — every value looks like a point on a temperature scale rather than an emotional one.

**The midpoint is a claim:**
A diverging scale with its midpoint at 0 says: "zero means neutral." If your data skews positive (most scores above zero), shifting the midpoint upward could make the map look more balanced — or more negative — without changing a single data value.

> 💡 **Questions to consider:**
> - Who cannot read the RdYlGn map? What is your responsibility to that reader?
> - At what point does choosing a flattering color scale become misleading?
> - Should "neutral" be at the center of the color scale, or at the center of your data's range?

In [ ]:
# Decision 6: Color scale — four options compared

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    size_max=40, map_style="carto-positron",
    center={"lat": 37.5, "lon": -78.0}, zoom=6, height=440
)

scales = [
    ("RdYlGn  — default diverging (red=bad, green=good, accessibility issues)", "RdYlGn",  0),
    ("RdBu_r  — diverging, more accessible, less culturally loaded",            "RdBu_r",  0),
    ("Viridis — colorblind-safe, perceptually uniform, but loses pos/neg frame","Viridis", None),
    ("Spectral — high-contrast diverging",                                      "Spectral", 0),
]

for title, scale, midpoint in scales:
    kw = {"color_continuous_midpoint": midpoint} if midpoint is not None else {}
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        color_continuous_scale=scale,
        title=title, **base, **kw
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0))
    fig.show()

print("💡 Which scale changes your emotional reaction to the map most? "
      "Does that reaction reflect the data — or the colors?")


### Decision 7: Base Map Style

The base map provides visual context — but it also makes implicit claims about geographic precision that the geoparsed data may not support.

| Style | Message it sends | Risk |
|---|---|---|
| Minimal / light (carto-positron) | Focus is on the data; geography is background | Strips context that might help readers orient themselves |
| Dark (carto-darkmatter) | High contrast; sentiment colours pop | Can make the map feel ominous regardless of the actual values |
| Full street map (OpenStreetMap) | Rich geographic context | Roads and labels compete with the data; implies more precision than geoparsing provides |

> 💡 **Questions to consider:**
> - Does a dark base map change how you *feel* about the sentiment data even before reading a value?
> - At what point does a stylistic choice become a rhetorical one?

In [ ]:
# Decision 7: Base map style

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    size_max=40, height=430
)

print("── Base map style: three options (same data, same zoom) ──")
for label, style in [
    ("Light — carto-positron  (minimal, data-focused)",    "carto-positron"),
    ("Dark — carto-darkmatter  (high contrast, dramatic)", "carto-darkmatter"),
    ("Standard — open-street-map  (maximum context)",      "open-street-map"),
]:
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        map_style=style, center={"lat": 37.5, "lon": -78.0}, zoom=6,
        title=label, **base
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()

print("\n💡 Does the dark base map change how you feel about the data before reading a value?")


### Decision 8: Zoom & Viewport

Zoom level is a framing decision — it determines what is *in frame* and what is cut off. Zoom out to national scale and you will see geoparser errors: posts resolved to Paris, London, or cities in Asia. Zoom into Harrisonburg and those errors are out of frame — which makes the map cleaner, but also hides the fact that those errors exist.

> 💡 **Questions to consider:**
> - At national zoom, do any locations look obviously wrong?
> - Is choosing a tight zoom that hides geoparser errors honest? What should you disclose in a caption?
> - What geographic extent best serves your research question?

In [ ]:
# Decision 8: Zoom & viewport

base = dict(
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    size_max=40, height=430,
    map_style="carto-positron"
)

print("── Zoom & viewport: three framing choices ──")
for zoom, lat, lon, label in [
    (3,  38.0,  -96.0,  "National (zoom=3) — geoparser errors are now visible"),
    (6,  37.5,  -78.0,  "Virginia (zoom=6) — regional framing"),
    (11, 38.44, -78.87, "Harrisonburg (zoom=11) — campus-level detail"),
]:
    fig = px.scatter_map(
        df_reddit_place_sentiments,
        center={"lat": lat, "lon": lon}, zoom=zoom,
        title=label, **base
    )
    fig.update_layout(margin=dict(r=0, t=50, l=0, b=0), coloraxis_showscale=False)
    fig.show()


### 6.2 Design Brief

**Fill in this table before you write a single line of code.** A design brief forces every parameter to be a deliberate choice rather than an accepted default. Reference your observations from Decisions 1–8.

| Design Dimension | Your Choice | Reasoning |
|---|---|---|
| **Filtering threshold** | min post count: ___ | |
| **Place type filter** | Which types to include: ___ | |
| **Bubble size encoding** | Raw count / √count / uniform | |
| **Size classification** | Equal interval / Quantile / Jenks · Classes: ___ | |
| **Sentiment bucketing** | Continuous or categorical? If categorical, threshold: ___ | |
| **Color scale** | | |
| **Base map style** | | |
| **Center coordinates** | Lat: ___ · Lon: ___ | |
| **Zoom level** | | |
| **Hover information** | What does a reader need to see? | |

> ⚠️ **Rule:** Your submitted map must differ from the reference implementation in at least three deliberate ways, each justifiable from this brief.

In [ ]:
# ============================================================
# YOUR FINAL MAP — build from your design brief above
# Every parameter must match a decision in your brief.
# ============================================================

fig = px.scatter_map(
    df_reddit_place_sentiments,
    lat="latitude",
    lon="longitude",
    # TODO: complete your design
)

fig.update_layout(
    # TODO: layout adjustments
)

fig.show()

### 6.3 Reference Implementation

The map below is one possible design using reasonable defaults. Study it critically before you submit your own version: which of the five decisions does it make, and are those the right choices for your research question?

After reviewing it, return to your own map cell above and revise where needed.

In [ ]:
# Reference implementation — study this critically, then go back and refine your own map above.
# You may NOT submit this cell's output unchanged as your final map.

fig = px.scatter_map(
    df_reddit_place_sentiments,
    lat="latitude", lon="longitude",
    size="location_count",
    color="avg_roberta_compound",
    hover_name="place",
    hover_data={'location_count': True, 'avg_roberta_compound': ':.3f', 'latitude': False, 'longitude': False},
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    size_max=50,
    title='Interactive Sentiment Map<br><sub>Bubble size = post count, Color = sentiment (red=negative, green=positive)</sub>',
    map_style="carto-positron",
    center={"lat": 37.5246322, "lon": -77.5758331},
    zoom=6, height=700
)
fig.update_traces(
    hovertemplate="<b>%{hovertext}</b><br>Posts: %{customdata[0]}<br>Avg Sentiment: %{customdata[1]}<br><extra></extra>"
)
fig.update_layout(
    margin={"r": 0, "t": 70, "l": 0, "b": 0},
    coloraxis_colorbar=dict(
        title="Average Sentiment",
        tickvals=[-0.5, -0.25, 0, 0.25, 0.5],
        ticktext=["Very Negative", "Negative", "Neutral", "Positive", "Very Positive"]
    )
)
fig.show()

print(f"📍 Total locations mapped: {len(df_reddit_place_sentiments)}")
print(f"📊 Total posts represented: {df_reddit_place_sentiments['location_count'].sum()}")
if len(df_reddit_place_sentiments) > 0:
    most_discussed = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['location_count'].idxmax()]
    most_positive  = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['avg_roberta_compound'].idxmax()]
    most_negative  = df_reddit_place_sentiments.loc[df_reddit_place_sentiments['avg_roberta_compound'].idxmin()]
    print(f"🔥 Most discussed: {most_discussed['place']} ({most_discussed['location_count']} posts)")
    print(f"😊 Most positive:  {most_positive['place']} (sentiment: {most_positive['avg_roberta_compound']:.3f})")
    print(f"😞 Most negative:  {most_negative['place']} (sentiment: {most_negative['avg_roberta_compound']:.3f})")

📍 Total locations mapped: 522
📊 Total posts represented: 1785
🔥 Most discussed: City of Harrisonburg (190 posts)
😊 Most positive:  Planetarium (sentiment: 0.983)
😞 Most negative:  Hampton Roads (sentiment: -0.839)


> 💡 **Critical Reflection:**
> - What patterns do you see geographically? Are certain regions consistently more positive or negative?
> - Find a location you know. Does the sentiment match your expectation? If not, what in the pipeline might explain it — data cleaning, NER, geoparsing, or the sentiment model?
> - What would the map look like if you used VADER scores instead of RoBERTa? What would be different?
> - This map represents what *Reddit users wrote about these places*, not what the places are actually like. What is the difference, and why does it matter?


## Lesson Summary

Here is what you covered in this lesson — and in the unit as a whole:

### The Full Pipeline
| Step | Lesson | Tool | What it produced |
|------|--------|------|-----------------|
| Load & clean data | 3 | Pandas | Sentence-level DataFrame |
| Extract place names | 4 | spaCy NER + transformer NER | Entity spans per sentence |
| Resolve to coordinates | 4 | Geoparser + GeoNames | Latitude/longitude per mention |
| Score sentiment | 5.1 | VADER | `vader_sentiment` per sentence |
| Score sentiment | 5.2 | RoBERTa | `roberta_compound` per sentence |
| Aggregate & map | 6 | Plotly | Emotional geography |

### Key Concepts
- **Named Entity Recognition (NER)** — identifying place names (and other entities) in unstructured text
- **Geoparsing** — disambiguating place names and resolving them to geographic coordinates
- **Sentiment analysis** — scoring text on a positive/negative scale; rule-based (VADER) vs. transformer-based (RoBERTa)
- **Aggregation** — collapsing row-level data to a summary by group (here: by place)
- **Limitations compound** — every imperfect step in a pipeline introduces noise that accumulates in the final output

### What the Map Does — and Doesn't — Show
This map visualises *how Reddit users wrote about places*, filtered through several imperfect models. It is not a ground-truth measure of how happy or unhappy those places are. That distinction — between the signal in the data and the reality it is meant to represent — is at the heart of critical data literacy.

---

➡️ **Next:** [Project: Mapping Emotions](../project_mapping_emotions/README.md)
